### Feature engineering

In [16]:
from load import load_jobs

# `lf` and the label expressions are defined once in scripts/load.py, which is
# also what data-analysis.ipynb uses. This notebook can therefore be run on its own.
globals().update(load_jobs())
print(f"lf ready: {len(lf.collect_schema().names())} columns")


Found 168 Parquet partition files.
Total dataset size on disk: 15.60 GB across 168 files.
Columns in NEED not present in raw dataset: ['Group']
Normalised 8 epoch-ms columns (0 -> null): QDate_ms, JobStartDate_ms, JobCurrentStartDate_ms, JobCurrentStartExecutingDate_ms, CompletionDate_ms, EnteredCurrentStatus_ms, LastMatchTime_ms, x509UserProxyExpiration_ms
Resolved: queue=QDate_ms start=JobStartDate_ms completion=CompletionDate terminal-status=EnteredCurrentStatus_ms
Lazy execution plan initialized successfully!
lf ready: 58 columns


In [17]:
import torch 

# --- Configuration & Split Windows ---

_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if _GPU else "cpu")
XGB_DEV = "cuda" if _GPU else "cpu"
CB_TASK = "GPU" if _GPU else "CPU"
LGBM_DEV = "cpu"

# Feature Schema
# CampaignId and CampaignStageId are dropped: CampaignName is strictly finer than
# CampaignId (475 name levels cover all 467 id levels, so nothing is lost), and
# CampaignStageName is the stage's kind rather than its instance. The ids are the
# highest-cardinality identity columns in the matrix and the ones that recur least
# after the cutoff -- CampaignStageId appears in only 82% of July test rows.
SUB_CAT = ["Group", "Owner", "PomsLauncher", "CampaignName", "CampaignStageName",
           "CampaignType", "TestLaunch", "JobsubGroup",
           "Image", "BlacklistSites"]
SUB_NUM = ["RequestCpus", "RequestDisk", "RequestMemory", "RequestSlots", 
           "ExecutableSize", "TransferInputMB", "ExpectedLifetime", "TotalSubmitProcs"]
MATCH_CAT = ["MatchSite", "MatchEntry", "MatchQueue", "MatchResource", "MatchCpus"]
MATCH_NUM = ["CpusProvisioned", "DiskProvisioned", "MemoryProvisioned"]
CAT_ALL = SUB_CAT + MATCH_CAT
NUM_ALL = SUB_NUM + MATCH_NUM

# Standard column mapping
CAND = {
    "Group": ["Group", "AccountingGroup"], "Owner": ["Owner"],
    "PomsLauncher": ["POMS_LAUNCHER", "POMS4_LAUNCHER"], "CampaignName": ["POMS4_CAMPAIGN_NAME"],
    "CampaignStageName": ["POMS4_CAMPAIGN_STAGE_NAME"],
    "CampaignType": ["DerivedCampaignType"],
    "TestLaunch": ["POMS4_TEST_LAUNCH"], "JobsubGroup": ["Jobsub_Group"],
    "Image": ["SingularityImage"], "BlacklistSites": ["Blacklist_Sites"],
    "MatchSite": ["MATCH_EXP_JOB_GLIDEIN_Site", "MATCH_GLIDEIN_Site"],
    "MatchEntry": ["MATCH_GLIDEIN_Entry_Name"], "MatchQueue": ["MATCH_GLIDEIN_SiteWMS_Queue"],
    "MatchResource": ["MachineAttrGLIDEIN_ResourceName0"], "MatchCpus": ["MachineAttrCpus0"],
    "Node": ["LastRemoteHost", "RemoteHost"],
    "RequestCpus": ["RequestCpus"], "RequestDisk": ["RequestDisk"], "RequestMemory": ["RequestMemory"],
    "RequestSlots": ["RequestSlots"], "ExecutableSize": ["ExecutableSize"],
    "TransferInputMB": ["TransferInputSizeMB"], "ExpectedLifetime": ["JOB_EXPECTED_MAX_LIFETIME"],
    "TotalSubmitProcs": ["TotalSubmitProcs"], "CpusProvisioned": ["CpusProvisioned"],
    "DiskProvisioned": ["DiskProvisioned"], "MemoryProvisioned": ["MemoryProvisioned"],
    "QDate": ["QDate_ms", "QDate"],
    "JobStart": ["JobStartDate_ms", "JobStartDate", "JobCurrentStartDate_ms", "JobCurrentStartDate"],
    "CompletionDate": ["CompletionDate_ms", "CompletionDate"],
}

### Deriving CampaignType from stage name

`POMS4_CAMPAIGN_TYPE` is null for every row in the raw data (100% -- verified
against all 32 partition files), including the ~25% of jobs that have real
`POMS4_CAMPAIGN_ID`/`NAME`/`STAGE_NAME` values. It was never populated
upstream; this isn't something earlier cells in this notebook broke.

Per the "Campaign Type" table in `FIFE-Docs.md`, the type can be recovered
from keywords in `POMS4_CAMPAIGN_STAGE_NAME` instead. Jobs with no POMS4_*
fields at all aren't part of any campaign -- these get `"User"`.

In [18]:
import polars as pl 

# Campaign-type keyword table, mirroring the "Campaign Type" table in
# FIFE-Docs.md. Edit here (and keep the doc in sync) to reclassify a
# stage-name substring.
CAMPAIGN_TYPE_KEYWORDS = {
    "Generation": ["gen", "dio", "endpoint", "corsika", "sim", "wiremod", "ly", "offset",
                   "spill", "g4", "beamgun", "decay", "surface", "cryo"],
    "Reconstruction": ["stage0", "stage1", "reco", "reco1", "reco2", "digi", "track", "decode",
                       "recluster", "fullproduction", "ndlar", "compress", "convert", "fmatch"],
    "Merging": ["merge", "skim", "hadd", "filter", "scrub", "watchdog", "sleep", "test",
                "fclless", "concat", "mix"],
    "Analysis": ["ana", "caf", "ntuple", "larcv"],
}


def _classify_campaign_type(stage_name):
    """Composite stage names (e.g. 'gen_g4_detsim_reco1_reco2_caf') match
    keywords from several categories at once. We take the LAST (rightmost)
    match, on the assumption that the terminal step is what the stage
    actually delivers -- that example ends in 'caf' -> Analysis, even though
    it runs gen/reco steps first. This tie-break decides ~2.98M jobs across
    45 composite stage names, so it's worth a sanity check against how
    these have been labeled by hand in the past.
    """
    low = stage_name.lower()
    best_cat, best_pos = None, -1
    for cat, kws in CAMPAIGN_TYPE_KEYWORDS.items():
        for kw in kws:
            pos = low.find(kw)
            if pos > best_pos:
                best_pos, best_cat = pos, cat
    return best_cat or "Unmapped"


# Build the lookup once over the distinct stage names (~164), then map the
# whole column against it -- far cheaper than a per-row Python call.
_distinct_stages = (
    lf.select(pl.col("POMS4_CAMPAIGN_STAGE_NAME").unique())
    .collect()["POMS4_CAMPAIGN_STAGE_NAME"].drop_nulls().to_list()
)
_stage_to_type = {s: _classify_campaign_type(s) for s in _distinct_stages}

# No POMS4 fields at all -> not part of a campaign -> a user job.
lf = lf.with_columns(
    pl.when(pl.col("POMS4_CAMPAIGN_STAGE_NAME").is_null())
      .then(pl.lit("User"))
      .otherwise(
          pl.col("POMS4_CAMPAIGN_STAGE_NAME").replace_strict(_stage_to_type, default="Unmapped")
      )
      .alias("DerivedCampaignType")
)

# Register the derived column in `have`, the same way cell 2 does for
# "Group". `have` is what CAND/cat_expr() gate on -- a column present in
# `lf` but missing from `have` is silently replaced by a constant 0, which
# is exactly the all-null CampaignType failure this cell exists to fix.
if "DerivedCampaignType" not in have:
    have.append("DerivedCampaignType")

_diag = lf.group_by("DerivedCampaignType").len().sort("len", descending=True).collect()
_total = _diag["len"].sum()
print("CampaignType distribution:")
for row in _diag.iter_rows(named=True):
    print(f"  {row['DerivedCampaignType']:15s} {row['len']:>12,}  ({100 * row['len'] / _total:5.2f}%)")

# Recover the integer code each label ends up with. cat_expr() hashes the
# string, then cell 17 dense-ranks via np.unique -- which sorts -- so a
# label's final code is the rank of its hash among the hashes of the labels
# actually present. Built from _diag (observed values) rather than a fixed
# list, so it stays correct if a filter ever drops a type entirely.
_present = _diag["DerivedCampaignType"].to_list()
_hashes = (
    pl.DataFrame({"v": _present})
    .select(
        (pl.col("v").cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629)
        .cast(pl.Int32).alias("h")
    )["h"].to_list()
)
CAMPAIGN_TYPE_CODES = {
    rank: lab
    for rank, (lab, _) in enumerate(sorted(zip(_present, _hashes), key=lambda kv: kv[1]))
}
print("\nCampaignType code -> label (saved to schema_meta.json):")
for _code, _lab in CAMPAIGN_TYPE_CODES.items():
    print(f"  {_code} -> {_lab}")

_unmapped = sorted(s for s, t in _stage_to_type.items() if t == "Unmapped")
assert not _unmapped, (
    f"{len(_unmapped)} stage name(s) matched no keyword: {_unmapped}. "
    f"Extend CAMPAIGN_TYPE_KEYWORDS above (and FIFE-Docs.md) to cover them."
)
print(f"\nAll {len(_distinct_stages)} distinct stage names mapped; "
      f"{_diag.height} unique CampaignType values.")


CampaignType distribution:
  User              48,063,579  (75.00%)
  Reconstruction     9,253,386  (14.44%)
  Analysis           5,237,589  ( 8.17%)
  Generation         1,462,791  ( 2.28%)
  Merging               71,013  ( 0.11%)

CampaignType code -> label (saved to schema_meta.json):
  0 -> Reconstruction
  1 -> Analysis
  2 -> User
  3 -> Merging
  4 -> Generation

All 164 distinct stage names mapped; 5 unique CampaignType values.


In [19]:
import os
import gc
import sys
import time
import datetime as _dt
import numpy as np
import polars as pl
import torch

def cat_expr(std):
    raw = R.get(std)
    if raw is None or raw not in have:
        return pl.lit(0).cast(pl.Int32).alias("c_" + std)
    return ((pl.col(raw).cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629)
            .cast(pl.Int32).alias("c_" + std))

def num_expr(std):
    raw = R.get(std)
    if raw is None or raw not in have:
        return pl.lit(0.0).cast(pl.Float32).alias("n_" + std)
    return pl.col(raw).cast(pl.Float32, strict=False).fill_null(0.0).alias("n_" + std)

R = {std: next((c for c in cands if c in have), None) for std, cands in CAND.items()}

sel_exprs = (
    [cat_expr(c) for c in CAT_ALL] + [cat_expr("Node")]
    + [num_expr(c) for c in NUM_ALL]
    + [to_sec(R["QDate"]).alias("t_q") if R["QDate"] else pl.lit(None).alias("t_q"),
       to_sec(R["JobStart"]).alias("t_s") if R["JobStart"] else pl.lit(None).alias("t_s"),
       to_sec(R["CompletionDate"]).alias("t_c") if R["CompletionDate"] else pl.lit(None).alias("t_c"),
       # Accumulated wall clock. Jobs that ran but were REMOVED never get a
       # CompletionDate, so start + walltime is the only way to date their outcome
       # for the temporal split. Not a training feature -- it is outcome-derived.
       (pl.col("RemoteWallClockTime").cast(pl.Float64, strict=False).alias("t_w")
        if "RemoteWallClockTime" in have else pl.lit(None, dtype=pl.Float64).alias("t_w"))]
    + [# A pilot is exactly Cmd == "./glidein_startup.sh". Pilots are the slots
       # workflow jobs wait for, so how many are running is the supply side of the
       # queue -- not a property of the job, but of what it is waiting on.
       ((pl.col("Cmd").cast(pl.Utf8).fill_null("") == "./glidein_startup.sh")
        .alias("is_pilot") if "Cmd" in have
        else pl.lit(False).alias("is_pilot"))]
    + [pl.col("Failed").cast(pl.Int8), pl.col("hw_fault").cast(pl.Int8),
       pl.col("fault_type").cast(pl.Int8), pl.col("wait_s").cast(pl.Float64),
       # Ran: this job actually started, so it is a valid E1/E3 target. Never-ran
       # jobs stay in the frame as queue context and are masked out downstream.
       pl.col("Ran").cast(pl.Int8),
       # When the job left the queue -- start time if it ran, terminal-status time
       # if it was removed first. Drives the idle-queue-depth feature.
       pl.col("t_queue_exit").cast(pl.Float64)]
)

t0 = time.time()
# load.load_jobs already restricted `lf` to the window starting WINDOW_START.
print(f"window starts {WINDOW_START} (QMIN_S={QMIN_S})")
df = lf.select(sel_exprs).collect()
ntot = df.height
print(f"Loaded {ntot:,} rows into RAM in {time.time() - t0:.1f}s")

# Extract categorical codes
codes = np.column_stack([df["c_" + c].to_numpy() for c in CAT_ALL])
node_k = df["c_Node"].to_numpy()
cards = []
for j in range(codes.shape[1]):
    u, inv = np.unique(codes[:, j], return_inverse=True)
    codes[:, j] = inv.astype(np.int32)
    cards.append(len(u))

# Extract numerical features. Standardisation is deferred until the splits are
# built, because the scaler must not see the test period -- see below.
X_num = np.column_stack([df["n_" + c].to_numpy() for c in NUM_ALL]).astype(np.float32)

# Timestamps & Labels
qs = df["t_q"].to_numpy()
jst = df["t_s"].to_numpy()
js = jst  # Alias for start time
comp = df["t_c"].to_numpy()
wall = df["t_w"].to_numpy()
ran = df["Ran"].to_numpy().astype(bool)
is_pilot = df["is_pilot"].to_numpy().astype(bool)
qexit = df["t_queue_exit"].to_numpy()
wait_sv = df["wait_s"].to_numpy()
ftype = df["fault_type"].to_numpy()
failed = df["Failed"].to_numpy().astype(np.int8)
hw = df["hw_fault"].to_numpy().astype(np.int8)

# Evaluation splits, from eval/splits.py -- the same function eval/dataset.py calls.
#
# The cut is on when each label BECOMES OBSERVABLE, not when the job was submitted:
# wait time is known at execution start, failure and attribution only at termination.
# E2 therefore gets an earlier observation time than E1/E3, and its own splits.
from eval.helper import terminal_time, temporal_masks
from eval.splits import (build_splits, describe as _describe_splits,
                         SPLIT_DESIGN as SPLIT_DESIGN_USED,
                         OOT_FRACTION as OOT_FRACTION_USED)

CUTOFF = _dt.datetime(2025, 7, 1, tzinfo=_dt.timezone.utc)
CUT_EPOCH = CUTOFF.timestamp()

SPLITS, TAU = {}, {}
for _exp, _label_src in (("e2", jst), ("e1e3", comp)):
    TAU[_exp], _ = terminal_time(_label_src, job_start=jst, wall_clock=wall,
                                 qdate=qs, floor=QMIN_S)
    _tr_t, _te_t, _ = temporal_masks(qs, CUT_EPOCH, label_time=TAU[_exp], verbose=True)
    SPLITS[_exp] = build_splits(np.where(_tr_t)[0], np.where(_te_t)[0],
                                order=TAU[_exp], n_rows=ntot, verbose=True)

# Standardise now, with mean and sd taken from PRE-CUTOFF rows only. Fitting the
# scaler on the whole matrix let test-period statistics into every training feature.
# No labels are involved so the leak is weak, but it is the same class of thing this
# work is about. Pre-cutoff is also identical for both arms: the temporal arm sees
# exactly its own training statistics, and the random arm sees less than it is
# entitled to, never more. Trees are unaffected either way -- standardising is a
# monotonic transform -- but the neural models need the scale.
_fit_rows = SPLITS["e2"]["temporal"][0]
for _j in range(X_num.shape[1]):
    _col = X_num[_fit_rows, _j]
    _mu = float(np.nanmean(_col))
    _sd = float(np.nanstd(_col)) or 1.0
    X_num[:, _j] = (X_num[:, _j] - _mu) / _sd
print(f"standardised {X_num.shape[1]} numeric columns on "
      f"{len(_fit_rows):,} pre-cutoff rows")

# Downstream cells use these names; they are the E2 honest arm.
tr_mask = np.zeros(ntot, dtype=bool); tr_mask[SPLITS["e2"]["temporal"][0]] = True
te_mask = np.zeros(ntot, dtype=bool); te_mask[SPLITS["e2"]["temporal"][1]] = True

# The polars frame is not used past this point -- every column needed downstream is
# already a numpy array. Holding it alongside Xmatch/Xsub is several GB of nothing.
del df
gc.collect()

print(f"\n{ntot:,} total rows")
for _exp, _sp in SPLITS.items():
    for _row in _describe_splits(_sp, qs=qs, cutoff_epoch=CUT_EPOCH):
        print(f"  {_exp:5s} {_row['split']:9s}/{_row['part']:5s} {_row['n']:>12,}"
              + (f"  {_row.get('pct_post_cutoff', float('nan')):5.1f}% post-cutoff"
                 if 'pct_post_cutoff' in _row else ""))

window starts 2025-02-01 (QMIN_S=1738368000)
Loaded 64,088,358 rows into RAM in 203.4s
[temporal split] train 48,542,883 | test 15,545,475 | 8,936 boundary jobs moved to test | 0 rows fell back to queue time
[shared-oot split] pool 57,714,713 | temporal train 48,542,883 / test 9,171,830 (through 2025-08-01) | random same sizes, 15.9% of its training rows from the protocol period
[shared-oot split] out-of-time test 6,373,645 rows from 2025-08-01, scored by both arms, trained on by neither
[temporal split] train 48,529,277 | test 15,559,081 | 22,542 boundary jobs moved to test | 0 rows fell back to queue time
[shared-oot split] pool 57,709,135 | temporal train 48,529,277 / test 9,179,858 (through 2025-08-01) | random same sizes, 15.9% of its training rows from the protocol period
[shared-oot split] out-of-time test 6,379,223 rows from 2025-08-01, scored by both arms, trained on by neither
standardised 11 numeric columns on 48,542,883 pre-cutoff rows

64,088,358 total rows
  e2    random 

In [23]:
# Trailing windows, in seconds.
TRAIL_WINDOWS_BY_KEY = {
    "site_fail": [60, 3600],
    "site_hw":   [60, 3600],
    "camp_fail": [60, 3600],
    "node_fail": [3600, 86400],
    "node_hw":   [3600, 86400],
}

# entry_fail is dropped: GlideinWMS entries nest inside sites, so it measures almost
# the same thing as site_fail -- r = 0.995 (15m) and 0.996 (60m) on a 2M-row sample.
# The *_hw rates are kept despite near-zero univariate correlation with Failed: they
# carry the largest |r| with the HARDWARE target of any trailing feature (0.040 for
# trail60m_site_hw), which is small only because hardware faults are 0.8% prevalent,
# and they are the sole hardware-specific trailing signal E3 has.
TRAIL_NAMES = list(TRAIL_WINDOWS_BY_KEY)

# Flat (window, statistic) pairs, in column order.
TRAIL_PAIRS = [(w, nm) for nm in TRAIL_NAMES for w in TRAIL_WINDOWS_BY_KEY[nm]]

n_cat, n_num = len(CAT_ALL), len(NUM_ALL)
n_base = n_cat + n_num
n_sc, n_sn = len(SUB_CAT), len(SUB_NUM)

XMATCH_COLS = (CAT_ALL + [c + " (std)" for c in NUM_ALL]
               + ["sin_hour@match", "cos_hour@match", "sin_wday@match", "cos_wday@match"]
               + [f"trail{w // 60}m_{nm}" for w, nm in TRAIL_PAIRS]
               + ["log_site_running@match", "log_total_running@match"])

XSUB_COLS = (SUB_CAT + [c + " (std)" for c in SUB_NUM]
             + ["sin_hour@submit", "cos_hour@submit", "sin_wday@submit", "cos_wday@submit"]
             + ["log_idle_queue_depth", "log_camp_trail_wait", "log_total_running@submit",
                "log_pilots_running@submit"])

NXM, NXS = len(XMATCH_COLS), len(XSUB_COLS)

# Vectorized helper for trailing failure/hardware rates.
#
# Measured on the full 64,088,358-row matrix (RTX 4090), for one key and one window:
#
#     lexsort((ct, ck))               3.37s        GPU sort            0.17s
#     searchsorted, 64M queries       8.36s        GPU searchsorted    0.01s
#     gather outcome + cumsum         0.60s        GPU cumsum          0.02s
#
# searchsorted dominates and runs twice per (key, window) pair for the hi and lo
# bounds, so the whole cell is mostly numpy's single-threaded binary search. The GPU
# path does the same integer arithmetic and is verified equal to the numpy path.
# Set FIFE_TRAIL_DEVICE=cpu to force the original route.
import os as _os

# The GPU here is shared. Pick the device with the most free memory, and only if it
# clears a headroom threshold -- otherwise fall back to the CPU path, which is slower
# but always works. FIFE_TRAIL_DEVICE forces a choice.
_TRAIL_MIN_FREE = float(_os.environ.get("FIFE_TRAIL_MIN_FREE_GB", "2.0")) * 1e9
# Queries are sent in chunks so the transient allocation stays small regardless of
# how much of the GPU someone else is using. 4M x 8 bytes = 32 MB per chunk.
_TRAIL_CHUNK = int(_os.environ.get("FIFE_TRAIL_CHUNK", str(4_000_000)))


def _pick_trail_device():
    forced = _os.environ.get("FIFE_TRAIL_DEVICE")
    if forced:
        return forced
    if not torch.cuda.is_available():
        return "cpu"
    best, best_free = None, 0
    for d in range(torch.cuda.device_count()):
        try:
            free, _ = torch.cuda.mem_get_info(d)
        except Exception:
            continue
        if free > best_free:
            best, best_free = d, free
    if best is None or best_free < _TRAIL_MIN_FREE:
        print(f"  [trail] only {best_free / 1e9:.1f} GB free on any GPU "
              f"(need {_TRAIL_MIN_FREE / 1e9:.1f}); using CPU")
        return "cpu"
    return f"cuda:{best}"


_TRAIL_DEV = _pick_trail_device()
print(f"trailing_rate device: {_TRAIL_DEV}")

_TRAIL_CACHE = {}


def _sorted_key(comp_key):
    """Hold the sorted composite key on the device that will search it. Falls back to
    host memory if the GPU cannot take it."""
    if _TRAIL_DEV == "cpu":
        return comp_key
    try:
        return torch.as_tensor(comp_key, device=_TRAIL_DEV)
    except torch.cuda.OutOfMemoryError:
        print("  [trail] GPU could not hold the sorted key; using CPU for this key")
        return comp_key


def _searchsorted(sorted_key, queries):
    """Right-side searchsorted. 64M queries take 8.4s single-threaded in numpy and
    0.01s on a GPU, which is the cell's dominant cost -- but the query array is 512 MB,
    so it goes over in chunks rather than all at once."""
    if not torch.is_tensor(sorted_key):
        return np.searchsorted(sorted_key, queries, "right")
    out = np.empty(len(queries), dtype=np.int64)
    try:
        for a in range(0, len(queries), _TRAIL_CHUNK):
            b = min(a + _TRAIL_CHUNK, len(queries))
            q = torch.as_tensor(queries[a:b], device=sorted_key.device)
            out[a:b] = torch.searchsorted(sorted_key, q, right=True).cpu().numpy()
            del q
        return out
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("  [trail] GPU OOM during searchsorted; finishing on CPU")
        return np.searchsorted(sorted_key.cpu().numpy(), queries, "right")


def _cumsum0(v):
    """Prefix sum with a leading zero, in float64 -- float32 loses too much over 55M
    terms to reproduce the same rates.

    Deliberately on the CPU: it costs 0.6s against 0.02s on a GPU, and doing it there
    would hold another ~900 MB for a saving the searchsorted change dwarfs.
    """
    return np.concatenate([[0.0], np.cumsum(v)])


def trailing_rate(key, tref, tcomp, outcome, is_comp, window_s, cache_key=None,
                  max_window=None):
    """Leak-free trailing rate of `outcome` over `window_s` seconds, per key.

    Sorting the completed rows dominates the cost and is identical for every
    statistic computed on the same key, so it is cached under `cache_key`. The
    encoding uses `max_window` for its span rather than this call's window, which
    makes the sorted array window-independent and lets the window bounds be cached
    too. The prefix sum depends on the key and the outcome but not the window, so it
    is cached as well -- site_fail runs at 60 s and 3600 s and would otherwise gather
    and accumulate 55M values twice for the same answer.
    """
    n = len(key); w = int(window_s)
    mw = int(max_window if max_window is not None else w)
    _id = cache_key if cache_key is not None else id(key)

    _ck = ("sort", _id, mw)
    ent = _TRAIL_CACHE.get(_ck)
    if ent is None:
        ci = np.where(is_comp)[0]
        ck = np.asarray(key)[ci].astype(np.int64)
        ct = np.rint(np.asarray(tcomp)[ci]).astype(np.int64)
        if len(ct) == 0:
            _TRAIL_CACHE[_ck] = ()
            return np.full(n, np.nan, np.float32), np.zeros(n, np.float32)
        o = np.lexsort((ct, ck))
        ci, ck, ct = ci[o], ck[o], ct[o]
        t0 = int(ct.min()); span = int(ct.max()) - t0 + mw + 3
        ent = (ci, ct, t0, span, _sorted_key(ck * span + (ct - t0)))
        _TRAIL_CACHE[_ck] = ent
        del ck
    if ent == ():
        return np.full(n, np.nan, np.float32), np.zeros(n, np.float32)
    ci, ct_s, t0, span, comp_key = ent

    tref = np.asarray(tref); tcomp = np.asarray(tcomp); outcome = np.asarray(outcome)
    _bk = ("bounds", _id, mw, w)
    bnd = _TRAIL_CACHE.get(_bk)
    if bnd is None:
        kq = np.asarray(key).astype(np.int64)
        tq = np.where(np.isnan(tref), t0 - w - 10, np.rint(tref)).astype(np.int64)
        hi = _searchsorted(comp_key, kq * span + np.clip(tq - t0, -1, span - 2))
        lo = _searchsorted(comp_key, kq * span + np.clip(tq - w - t0, -1, span - 2))
        bnd = (hi, lo)
        _TRAIL_CACHE[_bk] = bnd
        del kq, tq
    hi, lo = bnd

    _sk = ("csum", _id, mw, outcome.dtype.str, int(outcome.sum()))
    csum = _TRAIL_CACHE.get(_sk)
    if csum is None:
        csum = _cumsum0(outcome[ci].astype(np.float64))
        _TRAIL_CACHE[_sk] = csum

    cnt = (hi - lo).astype(np.float64); ssum = csum[hi] - csum[lo]
    self_in = np.asarray(is_comp) & ~np.isnan(tref) & (tcomp <= tref) & (tcomp > tref - window_s)
    ssum[self_in] -= outcome[self_in]; cnt[self_in] -= 1
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = np.where(cnt > 0, ssum / np.maximum(cnt, 1), np.nan)
    return rate.astype(np.float32), cnt.astype(np.float32)

# Cyclical clock features helper
def cyc_of(t):
    d = pl.from_epoch(pl.Series(np.asarray(t)).cast(pl.Int64, strict=False), time_unit="s")
    h = d.dt.hour().to_numpy().astype(np.float64)
    dw = d.dt.weekday().to_numpy().astype(np.float64)
    return np.nan_to_num(np.column_stack([np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24),
                                          np.sin(2 * np.pi * dw / 7), np.cos(2 * np.pi * dw / 7)])).astype(np.float32)

print("Calculating trailing state features...")
comp_ok = ~np.isnan(np.asarray(comp)); H1 = 3600.0
fl64, hw64 = np.asarray(failed).astype(np.float64), hw.astype(np.float64)
jsf = np.where(np.isnan(np.asarray(js)), -1e18, np.asarray(js))
tref_m = np.where(np.isnan(np.asarray(js)), np.asarray(qs), np.asarray(js))

site_k = codes[:, CAT_ALL.index("MatchSite")]
# CampaignName replaces the dropped CampaignId as the campaign grouping key.
# It is strictly finer -- 475 name levels cover all 467 ids -- so the trailing
# campaign statistics are grouped at least as tightly as before.
camp_k = codes[:, CAT_ALL.index("CampaignName")]

# Keyed by statistic name, and iterated over TRAIL_PAIRS, so the rate order is the
# column-name order by construction. The previous cross product over all windows x
# all specs cannot express per-key windows -- it would compute 18 rates for 10 named
# columns and silently misalign them.
TRAIL_SPEC = {
    "site_fail": (site_k, jsf, fl64),
    "node_fail": (node_k, jsf, fl64),
    "site_hw":   (site_k, jsf, hw64),
    "node_hw":   (node_k, jsf, hw64),
    "camp_fail": (camp_k, qs,  fl64),
}

_MAXW = max(w for w, _ in TRAIL_PAIRS)
_KEYNAME = {"site_fail": "site", "site_hw": "site", "node_fail": "node",
            "node_hw": "node", "camp_fail": "camp"}

tr_all = []
for w, nm in TRAIL_PAIRS:
    kv, tv, ov = TRAIL_SPEC[nm]
    r, _ = trailing_rate(kv, tv, comp, ov, comp_ok, float(w),
                         cache_key=_KEYNAME[nm], max_window=_MAXW)
    tr_all.append(np.nan_to_num(r))

_named = [f"trail{w // 60}m_{nm}" for w, nm in TRAIL_PAIRS]
assert len(tr_all) == len(_named), (
    f"{len(tr_all)} trailing rates computed but {len(_named)} column names")

for _v in _TRAIL_CACHE.values():
    if isinstance(_v, tuple) and len(_v) == 5 and torch.is_tensor(_v[4]):
        del _v
_TRAIL_CACHE.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Calculating concurrency features...")
fin2 = ~np.isnan(np.asarray(jst)) & ~np.isnan(np.asarray(comp))
s2 = np.sort(np.asarray(jst)[fin2]); c2 = np.sort(np.asarray(comp)[fin2])

def running_total(t):
    r = (np.searchsorted(s2, t, "right") - np.searchsorted(c2, t, "right")).astype(np.float64)
    self_run = fin2 & (np.asarray(jst) <= t) & (np.asarray(comp) > t)
    r[self_run] -= 1
    return np.clip(r, 0, None)

def running_pilots(t):
    """Pilots occupying a slot at time `t`.

    Same construction as running_total but over pilots only, and without the
    self-exclusion term: a workflow job is never its own pilot, so there is nothing
    to subtract. This is the supply the job is queueing against.
    """
    m = fin2 & is_pilot
    if not m.any():
        return np.zeros(len(np.atleast_1d(t)), dtype=np.float64)
    ps = np.sort(np.asarray(jst)[m])
    pc = np.sort(np.asarray(comp)[m])
    return np.clip(np.searchsorted(ps, t, "right")
                   - np.searchsorted(pc, t, "right"), 0, None).astype(np.float64)


def running_by_key(key, t):
    ki = np.asarray(key).astype(np.int64)
    ks = ki[fin2]
    st = np.rint(np.asarray(jst)[fin2]).astype(np.int64)
    ct = np.rint(np.asarray(comp)[fin2]).astype(np.int64)
    t0 = int(min(st.min(), ct.min())); span = int(max(st.max(), ct.max())) - t0 + 3
    a_s = np.sort(ks * span + (st - t0)); a_c = np.sort(ks * span + (ct - t0))
    q = ki * span + np.clip(np.rint(t).astype(np.int64) - t0, -1, span - 2)
    r = (np.searchsorted(a_s, q, "right") - np.searchsorted(a_c, q, "right")).astype(np.float64)
    self_run = fin2 & (np.asarray(jst) <= t) & (np.asarray(comp) > t)
    r[self_run] -= 1
    return np.clip(r, 0, None)

run_site = running_by_key(site_k, tref_m)
run_tot_m = running_total(tref_m)
cycS = cyc_of(tref_m)

# Assemble Xmatch ON DISK.
#
# Xmatch (~11 GB) and Xsub (~7 GB) previously lived in RAM alongside the source
# arrays, peaking around 35 GB on a shared 125 GB box -- which is what killed the
# kernel when another user's job took memory at the wrong moment. Writing through an
# open_memmap keeps the output out of RAM and persists it in the same pass, so no
# separate save step is needed (there was not one: the matrices were never written).
import os

from load import WINDOW_START  # noqa: F401  (documents the window this matrix covers)

from eval.paths import DATA_ROOT as FEAT_DIR   # per-host, from config/paths.yaml
os.makedirs(FEAT_DIR, exist_ok=True)

print(f"Assembling Xmatch on disk -> {FEAT_DIR}/Xmatch.npy "
      f"({int(ntot) * NXM * 4 / 1e9:.1f} GB)...")
Xmatch = np.lib.format.open_memmap(
    os.path.join(FEAT_DIR, "Xmatch.npy"), mode="w+",
    dtype=np.float32, shape=(int(ntot), NXM))
Xmatch[:, :n_cat] = codes
Xmatch[:, n_cat:n_base] = X_num
Xmatch[:, n_base:n_base + 4] = cycS
for j in range(len(tr_all)):
    Xmatch[:, n_base + 4 + j] = tr_all[j]
Xmatch[:, NXM - 2] = np.log1p(run_site)
Xmatch[:, NXM - 1] = np.log1p(run_tot_m)
del cycS, tr_all, run_site, run_tot_m

# Queue depth & submission features
print("Calculating submission & queue features...")
# Idle queue depth: jobs queued before t, minus jobs that had LEFT the queue by t.
# A job leaves either by starting or by being removed while still idle, and the
# September 2025 extraction dates both (see t_queue_exit). Counting only jobs that
# eventually started -- which is what this did while no removal timestamp existed --
# undercounts contention by omitting the ~11% that were removed from the queue.
_qs = np.asarray(qs, dtype=np.float64); _qx = np.asarray(qexit, dtype=np.float64)
fin = np.isfinite(_qs) & np.isfinite(_qx)
q_fin = np.sort(_qs[fin]); s_fin = np.sort(_qx[fin])
idle = np.clip(np.searchsorted(q_fin, _qs, "right")
               - np.searchsorted(s_fin, _qs, "right"), 0, None).astype(np.float64)
wok = comp_ok & ~np.isnan(np.asarray(wait_sv))
campw, _ = trailing_rate(camp_k, qs, comp, np.nan_to_num(np.asarray(wait_sv)), wok, H1)
run_tot_q = running_total(np.asarray(qs))
run_pil_q = running_pilots(np.asarray(qs))
cycQ = cyc_of(qs)
print(f"pilots running at submit: median {np.median(run_pil_q):,.0f}, "
      f"max {run_pil_q.max():,.0f}")

# Assemble Xsub on disk, same reasoning.
print(f"Assembling Xsub on disk -> {FEAT_DIR}/Xsub.npy "
      f"({int(ntot) * NXS * 4 / 1e9:.1f} GB)...")
Xsub = np.lib.format.open_memmap(
    os.path.join(FEAT_DIR, "Xsub.npy"), mode="w+",
    dtype=np.float32, shape=(int(ntot), NXS))
Xsub[:, :n_sc] = codes[:, :n_sc]
Xsub[:, n_sc:n_sc + n_sn] = X_num[:, :n_sn]
Xsub[:, n_sc + n_sn:n_sc + n_sn + 4] = cycQ
Xsub[:, -4] = np.log1p(idle)
Xsub[:, -3] = np.log1p(np.nan_to_num(campw))
Xsub[:, -2] = np.log1p(run_tot_q)
Xsub[:, -1] = np.log1p(run_pil_q)
del cycQ, idle, campw, run_tot_q, run_pil_q, q_fin, s_fin, jsf, fl64, hw64
gc.collect()

# Zero-copy views
X = Xmatch[:, :n_base]
trail = Xmatch[:, n_base + 4:NXM - 2]
cyc_q = Xsub[:, n_sc + n_sn:n_sc + n_sn + 4]

# Flush both to disk before anything else runs. Without this the pages are dirty in
# the page cache and a later crash loses them silently.
Xmatch.flush(); Xsub.flush()

print(f"Xmatch shape: {Xmatch.shape}  (base {n_base} + cyc 4 + trailing {len(TRAIL_PAIRS)} + concurrency 2)")
print(f"Xsub shape:   {Xsub.shape}  (sub cat {n_sc} + sub num {n_sn} + cyc 4 + queue state 3)")
print(f"written to {FEAT_DIR}")
for _f in ("Xmatch.npy", "Xsub.npy"):
    _p = os.path.join(FEAT_DIR, _f)
    print(f"  {_f:12s} {os.path.getsize(_p) / 1e9:6.2f} GB")

trailing_rate device: cuda:1
Calculating trailing state features...
Calculating concurrency features...
Assembling Xmatch on disk -> /media/data/allison/fife/Xmatch.npy (10.8 GB)...
Calculating submission & queue features...
pilots running at submit: median 174, max 3,542
Assembling Xsub on disk -> /media/data/allison/fife/Xsub.npy (6.7 GB)...
Xmatch shape: (64088358, 42)  (base 26 + cyc 4 + trailing 10 + concurrency 2)
Xsub shape:   (64088358, 26)  (sub cat 10 + sub num 8 + cyc 4 + queue state 3)
written to /media/data/allison/fife
  Xmatch.npy    10.77 GB
  Xsub.npy       6.67 GB


### Saving full data (anonymized)

In [6]:
ANON_START, ANON_END = "2025-02-01", "2025-09-01"      # [start, end)
ANON_DIR = "/media/storage0/allison/FIFE-Batch-Queues-anon"
ANON_OUT = f"{ANON_DIR}/fife_anon_{ANON_START}_{ANON_END}.parquet"

ANON_LABEL = {                      # column -> pseudonym prefix
    "Owner": "user", "AccountingGroup": "acct",
    "POMS4_CAMPAIGN_ID": "campaign", "POMS4_CAMPAIGN_NAME": "campname",
    "POMS4_CAMPAIGN_STAGE_NAME": "stage", "POMS4_CAMPAIGN_STAGE_ID": "stageid",
    "Jobsub_Group": "jgroup", "SingularityImage": "image", "Blacklist_Sites": "blacklist",
}
ANON_DENSE = ["LastRemoteHost", "ClusterId"]     # too many distinct values to label
# hashed.
ANON_KEEP = ["MATCH_EXP_JOB_GLIDEIN_Site", "MATCH_EXP_JOB_Site", "MATCH_GLIDEIN_Entry_Name",
             "MATCH_GLIDEIN_SiteWMS_Queue", "MachineAttrGLIDEIN_ResourceName0",
             "DerivedCampaignType", "Group"]
# Free text that embeds a username. The label logic matches on substrings of these
# ("condor_rm", "by user", "held 14 days", ...), so only the name is redacted -- the
# pattern the rules key on survives.
ANON_SCRUB = ["RemoveReason", "LastHoldReason"]

_cols = set(lf.collect_schema().names())

# Pilot flag. A glidein pilot is exactly Cmd == "./glidein_startup.sh"
if "Cmd" in _cols:
    _lf_src = lf.with_columns(
        (pl.col("Cmd") == "./glidein_startup.sh").fill_null(False).alias("IsPilot")
    ).drop("Cmd")
    _cols = set(_lf_src.collect_schema().names())
else:
    _lf_src = lf
    print("  WARNING: Cmd absent from lf -- IsPilot not written. Re-run the raw scan "
          "cell (de36b727) so Cmd is carried through.")

def _anon_hash(c):
    """cat_expr()'s hash -- same seed, so the ranking below matches Xmatch/Xsub."""
    return (pl.col(c).cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629).cast(pl.Int64)

_label_cols = [c for c in ANON_LABEL if c in _cols]
_dense_cols = [c for c in ANON_DENSE if c in _cols]
_lf_anon = _lf_src.with_columns([_anon_hash(c).alias(c) for c in _label_cols + _dense_cols])

# One pass to collect every column's sorted distinct hashes, then map to pseudonyms.
if _label_cols:
    _uniq = _lf_anon.select(
        [pl.col(c).unique().sort().implode().alias(c) for c in _label_cols]
    ).collect()
    _maps = {}
    for c in _label_cols:
        _vals = _uniq[c][0].to_list()
        # 0-based so the label matches the feature-matrix code exactly:
        # Xmatch/Xsub code k IS <prefix>k, no offset.
        _maps[c] = {h: f"{ANON_LABEL[c]}{i}" for i, h in enumerate(_vals)}
    _lf_anon = _lf_anon.with_columns(
        [pl.col(c).replace_strict(_maps[c], default=None).alias(c) for c in _label_cols]
    )

# High-cardinality columns: dense rank only, no label.
for c in _dense_cols:
    _lf_anon = _lf_anon.with_columns(
        pl.col(c).rank("dense").cast(pl.Int32).alias(c)
    )

_scrubbed = [c for c in ANON_SCRUB if c in _cols]
for c in _scrubbed:
    _lf_anon = _lf_anon.with_columns(
        pl.col(c).cast(pl.Utf8)
          .str.replace_all(r"(?i)(by user )\S+", r"${1}<redacted>")
          .str.replace_all(r"(?i)(user )[A-Za-z0-9._-]+( has removed)", r"${1}<redacted>${2}")
          .alias(c)
    )

# Restrict to the requested submission window.
_m0 = _dt.datetime.fromisoformat(ANON_START).replace(tzinfo=_dt.timezone.utc)
_m1 = _dt.datetime.fromisoformat(ANON_END).replace(tzinfo=_dt.timezone.utc)
_lf_anon = _lf_anon.filter(
    (to_sec(qcol) >= _m0.timestamp()) & (to_sec(qcol) < _m1.timestamp())
)

os.makedirs(os.path.dirname(ANON_OUT), exist_ok=True)
_lf_anon.sink_parquet(ANON_OUT, compression="zstd", compression_level=10)

print(f"Wrote {ANON_OUT}")
print(f"  window  : [{_m0:%Y-%m-%d} .. {_m1:%Y-%m-%d})")
print(f"  labelled: " + ", ".join(f"{c}->{ANON_LABEL[c]}N" for c in _label_cols))
print(f"  dense   : {', '.join(_dense_cols)}  (too many distinct values to label)")
print(f"  NOTE    : Xmatch/Xsub code k == <prefix>k here (0-based, exact match)")
print(f"  scrubbed: {', '.join(_scrubbed)}")
print(f"  readable: {', '.join(c for c in ANON_KEEP if c in _cols)}")

_chk = pl.scan_parquet(ANON_OUT)
print(f"  rows    : {_chk.select(pl.len()).collect().item():,}")
print(f"  size    : {os.path.getsize(ANON_OUT) / 1e6:,.1f} MB")
# Fail loudly if anything identifying survived.
_leaks = []
_csch = _chk.collect_schema()
for c in _label_cols:
    _sample = _chk.select(pl.col(c)).drop_nulls().head(1).collect()[c].to_list()
    if _sample and not str(_sample[0]).startswith(ANON_LABEL[c]):
        _leaks.append(f"{c} holds {_sample[0]!r}, not a {ANON_LABEL[c]}N pseudonym")
for c in _dense_cols:
    if _csch.get(c) != pl.Int32:
        _leaks.append(f"{c} is {_csch.get(c)}, expected dense Int32")
for c in _scrubbed:
    # Polars' regex engine has no look-around, so count-and-subtract instead.
    _tot = _chk.filter(pl.col(c).str.contains(r"(?i)by user ")).select(pl.len()).collect().item()
    _red = _chk.filter(pl.col(c).str.contains(r"(?i)by user <redacted>")).select(pl.len()).collect().item()
    if _tot - _red:
        _leaks.append(f"{c} still has {_tot - _red:,} un-redacted 'by user' values")
print("  CHECK   : " + ("OK, no identifying values found" if not _leaks else "LEAKS -> " + "; ".join(_leaks)))

Wrote /media/storage0/allison/FIFE-Batch-Queues-anon/fife_anon_2025-02-01_2025-09-01.parquet
  window  : [2025-02-01 .. 2025-09-01)
  labelled: Owner->userN, AccountingGroup->acctN, POMS4_CAMPAIGN_ID->campaignN, POMS4_CAMPAIGN_NAME->campnameN, POMS4_CAMPAIGN_STAGE_NAME->stageN, POMS4_CAMPAIGN_STAGE_ID->stageidN, Jobsub_Group->jgroupN, SingularityImage->imageN, Blacklist_Sites->blacklistN
  dense   : LastRemoteHost, ClusterId  (too many distinct values to label)
  NOTE    : Xmatch/Xsub code k == <prefix>k here (0-based, exact match)
  scrubbed: RemoveReason, LastHoldReason
  readable: MATCH_EXP_JOB_GLIDEIN_Site, MATCH_EXP_JOB_Site, MATCH_GLIDEIN_Entry_Name, MATCH_GLIDEIN_SiteWMS_Queue, MachineAttrGLIDEIN_ResourceName0, DerivedCampaignType, Group
  rows    : 64,088,358
  size    : 2,429.8 MB
  CHECK   : OK, no identifying values found


### Saving training data for later runs (skip)

In [ ]:
import os 
import json
import numpy as np

# Same root the harness reads; override with FIFE_DATA_ROOT.
from eval.paths import DATA_ROOT as SAVE_DIR   # per-host, from config/paths.yaml
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Saving targets and split indices to {SAVE_DIR}")

# 1. Save target arrays individually for fast loading
np.save(os.path.join(SAVE_DIR, "failed.npy"), failed)
np.save(os.path.join(SAVE_DIR, "fault_type.npy"), ftype)
np.save(os.path.join(SAVE_DIR, "hw.npy"), hw)
# E1/E3 target mask: jobs that actually started. Never-ran rows remain in the
# feature matrices as queue context but are not valid prediction targets.
np.save(os.path.join(SAVE_DIR, "ran.npy"), ran)

# 2. Update the targets in the .npz file (without saving Xmatch/Xsub)
np.savez_compressed(
    os.path.join(SAVE_DIR, "targets_and_masks.npz"),
    failed=failed,
    hw=hw,
    wait_sv=wait_sv,
    tr_mask=tr_mask,
    te_mask=te_mask,
    qs=qs,
    jst=jst,
    comp=comp,
    wall=wall,
    ran=ran,
    qexit=qexit,
    fault_type=ftype
)

# 3. Persist the split indices themselves, not just boolean month masks. The loader
# rebuilds them from eval/splits.py, so these are a record of what was actually used
# -- and a check that the two agree.
# "oot" is a single shared index array, not a (train, test) pair, so it cannot go
# through the same unpacking as the two arms.
_split_payload = {}
for _exp, _sp in SPLITS.items():
    for _arm, _val in _sp.items():
        if _arm == "oot":
            _split_payload[f"{_exp}__oot"] = np.asarray(_val, dtype=np.int64)
        else:
            for _part, _idx in zip(("train", "test"), _val):
                _split_payload[f"{_exp}__{_arm}__{_part}"] = np.asarray(_idx, dtype=np.int64)
np.savez_compressed(os.path.join(SAVE_DIR, "splits.npz"),
                    design=np.asarray(SPLIT_DESIGN_USED), cutoff=np.asarray(CUT_EPOCH),
                    **_split_payload)
with open(os.path.join(SAVE_DIR, "splits_meta.json"), "w") as _f:
    json.dump({"design": SPLIT_DESIGN_USED, "cutoff": CUT_EPOCH,
               "oot_fraction": OOT_FRACTION_USED,
               "sizes": {k: int(len(v)) for k, v in _split_payload.items()}}, _f, indent=2)
print(f"Saved splits.npz: {', '.join(sorted(_split_payload))}")

# 4. Sanity check directly from disk
saved_failed = np.load(os.path.join(SAVE_DIR, "failed.npy"))
print(f"Saved failed.npy successfully.")
print(f"Total rows       : {len(saved_failed):,}")
print(f"Rows (all)       : {len(ran):,}   target population (Ran): {int(ran.sum()):,}")
print(f"Genuine Failures : {int((saved_failed[ran]==1).sum()):,} (within the target population)")
print(f"Hardware Faults  : {hw.sum():,} ({hw.sum()/max(saved_failed.sum(),1)*100:.2f}% of failures)")
_c, _j, _w = np.isfinite(comp), np.isfinite(jst), np.isfinite(wall)
print(f"Terminal time    : CompletionDate {_c.sum():,} | start+wall "
      f"{int((~_c & _j & _w).sum()):,} | neither {int((~_c & ~(_j & _w)).sum()):,}")

Saved splits.npz: e1e3__oot, e1e3__random__test, e1e3__random__train, e1e3__temporal__test, e1e3__temporal__train, e2__oot, e2__random__test, e2__random__train, e2__temporal__test, e2__temporal__train
Saved failed.npy successfully.
Total rows       : 64,088,358
Rows (all)       : 64,088,358   target population (Ran): 56,918,813
Genuine Failures : 8,603,725 (within the target population)
Hardware Faults  : 524,584 (5.81% of failures)
Terminal time    : CompletionDate 55,264,860 | start+wall 1,652,885 | neither 7,170,613


Saving metadata

In [25]:
metadata = {
    "XMATCH_COLS": XMATCH_COLS,
    "XSUB_COLS": XSUB_COLS,
    "cards": [int(c) for c in cards],
    "NCAT_MATCH": len(CAT_ALL),
    "NCAT_SUB": len(SUB_CAT),
    "n_base": n_base,
    # Per-key trailing windows, so a saved matrix records which
    # windows its trailing columns were built with.
    "TRAIL_WINDOWS_BY_KEY": TRAIL_WINDOWS_BY_KEY,
}

# Code -> label for CampaignType (JSON object keys must be strings). The
# codes come from a hash + dense-rank, so they carry no inherent meaning --
# without this map the column is unreadable downstream.
if "CAMPAIGN_TYPE_CODES" in dir():
    metadata["CAMPAIGN_TYPE_CODES"] = {str(k): v for k, v in CAMPAIGN_TYPE_CODES.items()}
else:
    print("[!] CAMPAIGN_TYPE_CODES not defined -- run the CampaignType derivation cell.")

with open(os.path.join(SAVE_DIR, "schema_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)